# Predicción sobre el conjunto de competición

**NIA 100522396**  
**NIA 100522401**

Este notebook carga el modelo final ya entrenado y lo utiliza para generar las predicciones sobre el conjunto de competición, guardando el resultado en el fichero `predicciones.csv`, tal y como pide el enunciado.

In [ ]:
import numpy as np
import pandas as pd
from joblib import load


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/P1_AA_datasets'
MODEL_PATH = f'{BASE_DIR}/modelo_final.joblib'
COMP_PATH = f'{BASE_DIR}/bank_competition.pkl'
PRED_PATH = f'{BASE_DIR}/predicciones.csv'

print('MODEL_PATH:', MODEL_PATH)
print('COMP_PATH:', COMP_PATH)
print('PRED_PATH:', PRED_PATH)

In [ ]:
def transform_pdays(X):
    X = X.copy()
    X["contacted_before"] = (X["pdays"] != -1).astype(int)
    X["pdays"] = X["pdays"].replace(-1, np.nan)
    return X


## Carga del modelo final y de los datos de competición

En primer lugar, se define la función auxiliar `transform_pdays`, necesaria para reconstruir correctamente la pipeline guardada. Después, se carga el modelo final previamente entrenado y guardado, junto con el dataset de competición sobre el que se deben realizar las predicciones.

In [ ]:
final_model = load(MODEL_PATH)
competition_data = pd.read_pickle(COMP_PATH)

print('Dimensiones del conjunto de competición:', competition_data.shape)
display(competition_data.head())

## Generación de predicciones

El modelo final se utiliza para predecir la variable `deposit` en el conjunto de competición. Después, las predicciones se convierten al formato exigido (`yes`/`no`) y se guardan en el archivo `predicciones.csv`.

In [ ]:
competition_pred = final_model.predict(competition_data)
competition_pred_labels = pd.Series(competition_pred).map({1: 'yes', 0: 'no'})

predicciones_df = pd.DataFrame({'deposit': competition_pred_labels})
predicciones_df.to_csv(PRED_PATH, index=False)

print(f'Predicciones guardadas en: {PRED_PATH}')
display(predicciones_df.head())
print(predicciones_df['deposit'].value_counts())

In [ ]:
print('Número de filas del conjunto de competición:', len(competition_data))
print('Número de filas en predicciones.csv:', len(predicciones_df))

assert len(competition_data) == len(predicciones_df)

Se ha comprobado que el modelo final puede cargarse correctamente y reutilizarse sobre nuevos datos. Además, las predicciones del conjunto de competición se han guardado en el fichero `predicciones.csv` con el formato solicitado en la práctica, y su número de filas coincide con el número de instancias del dataset de competición.